# 📂 Dataset Splitter — Train / Validation / Test

---

## 🎯 Purpose

When training any **Deep Learning** or **Machine Learning** model, we must divide our dataset into three parts:

| Split | Role | Why? |
|-------|------|------|
| **Train** | The model learns from this data | Largest portion — the model adjusts its weights using these samples |
| **Validation** | Used to tune hyper-parameters & monitor over-fitting | The model **never trains** on this — it's checked after each epoch |
| **Test** | Final, one-time evaluation of model performance | Completely unseen data to give an honest accuracy score |

---

## 📐 Split Strategy

The ideal ratio depends on how much data you have:

| Dataset Size | Train | Val | Test | Reasoning |
|---|---|---|---|---|
| **< 10,000** images | 70 % | 15 % | 15 % | Small dataset → need more val/test for reliable evaluation |
| **10,000 – 100,000** images | 80 % | 10 % | 10 % | Medium dataset → balanced split |
| **> 100,000** images | 90 % | 5 % | 5 % | Large dataset → even 5 % gives thousands of test samples |

---

## 🧰 Libraries Used

This notebook uses **only Python standard-library modules** — no external packages required:

| Module | Purpose |
|--------|----------|
| `os` | File-system traversal and path operations |
| `shutil` | Copying files from source to destination |
| `random` | Shuffling the file list for unbiased splits |
| `pathlib` | Modern, readable path handling |
| `math` | Rounding calculations for split indices |

> 💡 **No `pip install` needed!** This notebook runs on any Python 3.6+ environment.

---

## Step 1 — Import Standard Libraries

We import everything we need from Python's built-in modules.  
Nothing to install — these ship with every Python installation.

In [1]:
import os
import shutil
import random
import math
from pathlib import Path

print("✅ All standard libraries imported successfully.")

✅ All standard libraries imported successfully.


---

## Step 2 — Configure Paths

Set the **source** and **output** directories below.

### Expected Source Structure

Your `DATA_DIR` should be organised into **sub-folders**, one per class:

```
data/
├── class_A/
│   ├── img_001.jpg
│   ├── img_002.png
│   └── ...
├── class_B/
│   ├── img_001.jpg
│   └── ...
└── class_C/
    └── ...
```

### Configuration

| Variable | Description |
|----------|-------------|
| `DATA_DIR` | Path to your source data folder |
| `OUTPUT_DIR` | Where the split folders will be created |
| `SEED` | Random seed — same seed = same split every time (reproducibility) |

In [2]:
# ╔══════════════════════════════════════════════╗
# ║   📌  CHANGE THESE PATHS FOR YOUR PROJECT   ║
# ╚══════════════════════════════════════════════╝

DATA_DIR   = './data'          # ← source folder (class sub-folders inside)
OUTPUT_DIR = './split_data'    # ← destination for train / val / test
SEED       = 42                # ← random seed for reproducibility

# Supported image file extensions
IMAGE_EXTENSIONS = {
    '.jpg', '.jpeg', '.png', '.bmp', '.gif',
    '.webp', '.tiff', '.tif', '.svg',
}

print(f"📁 Source directory  : {os.path.abspath(DATA_DIR)}")
print(f"📦 Output directory  : {os.path.abspath(OUTPUT_DIR)}")
print(f"🎲 Random seed       : {SEED}")

📁 Source directory  : /Users/apple/Documents/Code/AI/Projects/DL/ai/notebooks/data
📦 Output directory  : /Users/apple/Documents/Code/AI/Projects/DL/ai/notebooks/split_data
🎲 Random seed       : 42


---

## Step 3 — Analyse the Dataset

Before doing anything, we **scan the source folder** to:

1. Discover all class sub-folders
2. Count how many images are in each class
3. Compute the **total image count** — this decides which split ratio to use

> ⚠️ **Important:** Files that don't match the supported image extensions are skipped automatically.

In [3]:
data_path = Path(DATA_DIR)

# ── Validate source directory exists ──
assert data_path.exists() and data_path.is_dir(), (
    f"❌ Source directory not found: {data_path.resolve()}\n"
    f"   Please check your DATA_DIR path in Step 2."
)

# ── Discover class folders (ignore hidden folders like .DS_Store) ──
classes = sorted([
    entry.name for entry in data_path.iterdir()
    if entry.is_dir() and not entry.name.startswith('.')
])

assert len(classes) > 0, (
    f"❌ No class sub-folders found in {data_path.resolve()}\n"
    f"   Make sure your data is organised into sub-folders (one per class)."
)

print(f"📁 Source       : {data_path.resolve()}")
print(f"📦 Classes found: {len(classes)}")
print(f"   → {classes}\n")

# ── Count images in each class ──
class_counts = {}   # { class_name: count }
total_images = 0

for cls_name in classes:
    cls_path = data_path / cls_name
    # Count only files with valid image extensions
    image_files = [
        f for f in cls_path.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    ]
    count = len(image_files)
    class_counts[cls_name] = count
    total_images += count
    print(f"   🏷️  {cls_name:20s} → {count:,} images")

print(f"\n{'━' * 45}")
print(f"   🖼️  Total images: {total_images:,}")
print(f"{'━' * 45}")

📁 Source       : /Users/apple/Documents/Code/AI/Projects/DL/ai/notebooks/data
📦 Classes found: 3
   → ['paper', 'rock', 'scissors']

   🏷️  paper                → 964 images
   🏷️  rock                 → 964 images
   🏷️  scissors             → 964 images

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   🖼️  Total images: 2,892
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


---

## Step 4 — Determine Split Ratios

Based on the **total image count** from Step 3, we automatically select the optimal split ratio.

### Why Different Ratios?

- **Small datasets (< 10K):** We need a larger validation and test set (15 % each) to get
  a statistically meaningful evaluation. With only a few thousand images, 5 % might be
  just a handful — not enough to judge the model fairly.

- **Medium datasets (10K – 100K):** A 10 % split gives thousands of samples for val/test,
  which is plenty for reliable metrics.

- **Large datasets (> 100K):** Even 5 % yields 5,000+ images — more than enough.
  We maximise training data to let the model learn better.

In [4]:
# ── Select split ratio based on dataset size ──
if total_images < 10_000:
    train_ratio, val_ratio, test_ratio = 0.70, 0.15, 0.15
    tier = "🟢 Small  (< 10K images)"
elif total_images < 100_000:
    train_ratio, val_ratio, test_ratio = 0.80, 0.10, 0.10
    tier = "🟡 Medium (10K – 100K images)"
else:
    train_ratio, val_ratio, test_ratio = 0.90, 0.05, 0.05
    tier = "🔴 Large  (> 100K images)"

# Sanity check: ratios must sum to 1.0
assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-9, "Ratios must sum to 1.0!"

print(f"📊 Dataset tier     : {tier}")
print(f"📐 Selected ratios  : Train {train_ratio:.0%}  |  Val {val_ratio:.0%}  |  Test {test_ratio:.0%}")
print()
print(f"   Expected counts (approximate):")
print(f"     🟩 Train : ~{int(total_images * train_ratio):,} images")
print(f"     🟧 Val   : ~{int(total_images * val_ratio):,} images")
print(f"     🟦 Test  : ~{int(total_images * test_ratio):,} images")

📊 Dataset tier     : 🟢 Small  (< 10K images)
📐 Selected ratios  : Train 70%  |  Val 15%  |  Test 15%

   Expected counts (approximate):
     🟩 Train : ~2,024 images
     🟧 Val   : ~433 images
     🟦 Test  : ~433 images


---

## Step 5 — Split the Dataset

This is the core step. For **each class**, we:

1. **List** all image files in the class folder
2. **Shuffle** them randomly (using our seed for reproducibility)
3. **Slice** the list into train / val / test portions
4. **Copy** the files into the output directory

### How the Splitting Works

```
Given 100 images and a 70/15/15 split:

  Shuffled list:  [img_42, img_07, img_91, img_23, ... img_56]
                   ├──────── 70 ────────┤├── 15 ──┤├── 15 ──┤
                          Train             Val       Test
```

> 🔀 **Why shuffle?** Without shuffling, the split might put all images of a certain
> pattern (e.g., alphabetically named) into the same set, causing **data leakage** or
> an unbalanced evaluation.

In [5]:
output_path = Path(OUTPUT_DIR)

# ── Clean previous output (if any) ──
if output_path.exists():
    shutil.rmtree(output_path)
    print(f"🗑️  Removed existing output directory: {output_path}")

# ── Create output directory structure ──
for split in ['train', 'val', 'test']:
    for cls_name in classes:
        (output_path / split / cls_name).mkdir(parents=True, exist_ok=True)

print(f"📁 Created output structure at: {output_path.resolve()}\n")

# ── Set random seed for reproducibility ──
random.seed(SEED)

# ── Split each class independently ──
split_counts = {'train': {}, 'val': {}, 'test': {}}  # track counts for verification

for cls_name in classes:
    cls_path = data_path / cls_name

    # Collect all image files in this class
    image_files = sorted([
        f for f in cls_path.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    ])

    # Shuffle the list
    random.shuffle(image_files)

    # Calculate split indices
    n = len(image_files)
    train_end = math.floor(n * train_ratio)
    val_end   = train_end + math.floor(n * val_ratio)
    # test gets the remainder → no images are lost due to rounding

    # Slice into three sets
    train_files = image_files[:train_end]
    val_files   = image_files[train_end:val_end]
    test_files  = image_files[val_end:]

    # Copy files to their destination
    for split_name, file_list in [('train', train_files), ('val', val_files), ('test', test_files)]:
        for src_file in file_list:
            dst_file = output_path / split_name / cls_name / src_file.name
            shutil.copy2(src_file, dst_file)  # copy2 preserves metadata
        split_counts[split_name][cls_name] = len(file_list)

    # Progress indicator
    print(
        f"   ✅ {cls_name:20s} → "
        f"Train: {len(train_files):,}  |  "
        f"Val: {len(val_files):,}  |  "
        f"Test: {len(test_files):,}"
    )

print(f"\n🎉 Splitting complete!")

📁 Created output structure at: /Users/apple/Documents/Code/AI/Projects/DL/ai/notebooks/split_data

   ✅ paper                → Train: 674  |  Val: 144  |  Test: 146
   ✅ rock                 → Train: 674  |  Val: 144  |  Test: 146
   ✅ scissors             → Train: 674  |  Val: 144  |  Test: 146

🎉 Splitting complete!


---

## Step 6 — Verify the Split

Always verify your split! We check that:

- ✅ Every image was copied (no data loss)
- ✅ The actual percentages match the intended ratios
- ✅ Each class is proportionally represented in every split

In [6]:
print("=" * 60)
print(f"{'SPLIT VERIFICATION REPORT':^60}")
print("=" * 60)

split_totals = {}

for split_name in ['train', 'val', 'test']:
    split_total = sum(split_counts[split_name].values())
    split_totals[split_name] = split_total
    pct = (split_total / total_images * 100) if total_images > 0 else 0

    print(f"\n📂 {split_name.upper():6s}  →  {split_total:,} images  ({pct:.1f}%)")
    for cls_name in classes:
        cnt = split_counts[split_name].get(cls_name, 0)
        print(f"     {cls_name:20s} : {cnt:,}")

# ── Grand total check ──
grand_total = sum(split_totals.values())
print(f"\n{'─' * 60}")
print(f"🧮 Grand total : {grand_total:,}  (original: {total_images:,})")

if grand_total == total_images:
    print("✅ Perfect — all images accounted for! No data lost.")
else:
    diff = abs(grand_total - total_images)
    print(f"⚠️  Mismatch! {diff} image(s) differ from original count.")

                 SPLIT VERIFICATION REPORT                  

📂 TRAIN   →  2,022 images  (69.9%)
     paper                : 674
     rock                 : 674
     scissors             : 674

📂 VAL     →  432 images  (14.9%)
     paper                : 144
     rock                 : 144
     scissors             : 144

📂 TEST    →  438 images  (15.1%)
     paper                : 146
     rock                 : 146
     scissors             : 146

────────────────────────────────────────────────────────────
🧮 Grand total : 2,892  (original: 2,892)
✅ Perfect — all images accounted for! No data lost.


---

## Step 7 — Output Directory Tree

A quick visual overview of what was created.

In [7]:
print(f"\n📁 Output structure:\n")
print(f"{OUTPUT_DIR}/")

splits = ['train', 'val', 'test']
for si, split_name in enumerate(splits):
    total = split_totals.get(split_name, 0)
    is_last_split = (si == len(splits) - 1)
    branch = '└──' if is_last_split else '├──'
    print(f"{branch} {split_name}/   ({total:,} images)")

    for ci, cls_name in enumerate(classes):
        cnt = split_counts[split_name].get(cls_name, 0)
        is_last_cls = (ci == len(classes) - 1)
        indent = '    ' if is_last_split else '│   '
        cls_branch = '└──' if is_last_cls else '├──'
        print(f"{indent}{cls_branch} {cls_name}/   ({cnt:,} images)")

print(f"\n🎉 Done! Your dataset is ready for training.")
print(f"   Use the paths below in your training script:\n")
print(f"   TRAIN_DIR = '{OUTPUT_DIR}/train'")
print(f"   VAL_DIR   = '{OUTPUT_DIR}/val'")
print(f"   TEST_DIR  = '{OUTPUT_DIR}/test'")


📁 Output structure:

./split_data/
├── train/   (2,022 images)
│   ├── paper/   (674 images)
│   ├── rock/   (674 images)
│   └── scissors/   (674 images)
├── val/   (432 images)
│   ├── paper/   (144 images)
│   ├── rock/   (144 images)
│   └── scissors/   (144 images)
└── test/   (438 images)
    ├── paper/   (146 images)
    ├── rock/   (146 images)
    └── scissors/   (146 images)

🎉 Done! Your dataset is ready for training.
   Use the paths below in your training script:

   TRAIN_DIR = './split_data/train'
   VAL_DIR   = './split_data/val'
   TEST_DIR  = './split_data/test'


---

## 📚 Key Concepts for Students

### Why Three Splits?

| Mistake | What Happens |
|---------|-------------|
| Training and testing on the **same** data | Model memorises answers → **inflated accuracy** that won't generalise |
| No validation set | You can't tune hyper-parameters without **overfitting to the test set** |
| No shuffling | Alphabetical or sequential ordering may group similar images together → **biased splits** |

### Why Use a Random Seed?

Setting `random.seed(42)` ensures that **every time you run this notebook, you get the exact same split**.
This is critical for:

- **Reproducibility** — other students/researchers can replicate your results
- **Fair comparison** — when comparing two models, both must be trained on identical data
- **Debugging** — if something goes wrong, you can recreate the exact same dataset

### Per-Class Splitting

We split **each class independently** rather than mixing everything together.
This guarantees that **class proportions are preserved** in every split
(also called **stratified splitting**).

```
Example: 60 rock + 40 paper = 100 images (60%/40% ratio)

  Train (70): 42 rock + 28 paper  → 60% / 40% ✅  (ratio preserved)
  Val   (15):  9 rock +  6 paper  → 60% / 40% ✅
  Test  (15):  9 rock +  6 paper  → 60% / 40% ✅
```

### Reusing This Notebook

This notebook works with **any image classification project**. Just:

1. Organise your images into class sub-folders
2. Update `DATA_DIR` in Step 2
3. Run all cells

The split ratio is chosen **automatically** based on your dataset size.